In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 1. Download dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "karagwaanntreasure/plant-disease-detection"
)

print("Dataset downloaded to:")
print(path)

# 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#3. Inspect dataset structure

In [ ]:
import os
print("Files/folders in dataset:")
for item in os.listdir(path):
    print(item)

In [ ]:
DATASET_PATH = os.path.join(path, "Dataset")
print("Dataset path:", DATASET_PATH)
print(os.listdir(DATASET_PATH))

# 3. Set paths
Change DATASET_PATH to wherever your dataset is located.

In [ ]:
DATASET_PATH = "/content/your_dataset_folder"
SAVE_PATH = "/content/drive/MyDrive/PlantDisease_XAI/processed"
os.makedirs(SAVE_PATH, exist_ok=True)
print("Dataset:", DATASET_PATH)
print("Save location:", SAVE_PATH)

#4. Count all classes

In [ ]:
class_counts = {}

for class_name in os.listdir(DATASET_PATH):
    class_path = os.path.join(DATASET_PATH, class_name)
    if os.path.isdir(class_path):
        count = 0
        for file in os.listdir(class_path):
            if file.lower().endswith(
                (".jpg", ".jpeg", ".png")
            ):
                count += 1
        class_counts[class_name] = count

class_counts = dict(
    sorted(
        class_counts.items(),
        key=lambda x: x[1]
    )
)

for class_name, count in class_counts.items():
    print(f"{class_name}: {count}")

# 5. Select disease

In [ ]:
TARGET_DISEASE = "Tomato_Early_blight"
HEALTHY_CLASS = "Tomato_healthy"

print("Target disease:", TARGET_DISEASE)
print("Healthy class:", HEALTHY_CLASS)

# 6. Get image paths

In [ ]:
def get_image_paths(folder):

    extensions = (
        ".jpg",
        ".jpeg",
        ".png"
    )

    return [
        os.path.join(folder, file)
        for file in os.listdir(folder)
        if file.lower().endswith(extensions)
    ]


disease_path = os.path.join(
    DATASET_PATH,
    TARGET_DISEASE
)

healthy_path = os.path.join(
    DATASET_PATH,
    HEALTHY_CLASS
)


disease_images = get_image_paths(disease_path)
healthy_images = get_image_paths(healthy_path)


print("Disease images:", len(disease_images))
print("Healthy images:", len(healthy_images))

In [ ]:
import random

random.seed(42)

# Shuffle both lists
random.shuffle(disease_images)
random.shuffle(healthy_images)

# Number of images in the minority class
num_images = len(disease_images)

# Keep the same number of healthy images
healthy_images = healthy_images[:num_images]

print("After balancing:")
print("Disease images:", len(disease_images))
print("Healthy images:", len(healthy_images))
print("Total images:", len(disease_images) + len(healthy_images))

# Create labels

In [ ]:
image_paths = disease_images + healthy_images

labels = (
    [1] * len(disease_images) +
    [0] * len(healthy_images)
)

print("Total images:", len(image_paths))
print("Total labels:", len(labels))

print("Disease labels:", labels.count(1))
print("Healthy labels:", labels.count(0))

# Dataset Split
70/15/15

In [ ]:
from sklearn.model_selection import train_test_split

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=42
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=42
)

print("Train:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))

# Define preprocessing

### Data Preprocessing

The Plant Disease dataset was obtained using KaggleHub and contains 23 different plant disease and healthy-leaf classes. To reduce computational complexity, the project focuses on **Tomato Early Blight** and **Tomato Healthy** classes. We selected 1,000 images from each class, resulting in a balanced dataset of 2,000 images.

The dataset was divided into **70% training, 15% validation, and 15% testing** using stratified sampling to maintain the same class distribution across all three subsets.

For image preprocessing, all images were converted to **RGB format** and resized to **224 × 224 pixels**. The images were then converted into PyTorch tensors and normalized using the standard **ImageNet mean and standard deviation** values:

- Mean: `[0.485, 0.456, 0.406]`
- Standard deviation: `[0.229, 0.224, 0.225]`

The final tensor shape for each image is **3 × 224 × 224**, representing the three RGB channels and the 224 × 224 spatial dimensions.

The processed training, validation, and testing tensors are saved as `.pt` files on Google Drive so that subsequent notebooks can directly load the processed data without repeating the preprocessing step.

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Preprocessing pipeline created.")

# preprocessing function

In [ ]:
from PIL import Image

def process_images(paths, labels):

    images = []
    processed_labels = []

    for path, label in zip(paths, labels):

        image = Image.open(path).convert("RGB")

        image = transform(image)

        images.append(image)
        processed_labels.append(label)

    images = torch.stack(images)

    labels = torch.tensor(
        processed_labels,
        dtype=torch.long
    )

    return images, labels

In [ ]:
X_train, y_train = process_images(
    train_paths,
    train_labels
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

In [ ]:
X_val, y_val = process_images(
    val_paths,
    val_labels
)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

In [ ]:
X_test, y_test = process_images(
    test_paths,
    test_labels
)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# Create Drive folder

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/XAI_Project/processed"
os.makedirs(SAVE_PATH, exist_ok=True)

print("Save path:", SAVE_PATH)

# Saving Dataset

In [ ]:
torch.save(
    {
        "images": X_train,
        "labels": y_train
    },
    os.path.join(SAVE_PATH, "train.pt")
)

print("train.pt saved")

In [ ]:
torch.save(
    {
        "images": X_val,
        "labels": y_val
    },
    os.path.join(SAVE_PATH, "val.pt")
)

print("val.pt saved")

In [ ]:
torch.save(
    {
        "images": X_test,
        "labels": y_test
    },
    os.path.join(SAVE_PATH, "test.pt")
)

print("test.pt saved")

In [ ]:
import json

In [ ]:
metadata = {
    "classes": {
        "0": HEALTHY_CLASS,
        "1": TARGET_DISEASE
    },
    "image_size": [224, 224],
    "channels": 3,
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    },
    "split": {
        "train": len(X_train),
        "validation": len(X_val),
        "test": len(X_test)
    }
}

with open(
    os.path.join(SAVE_PATH, "metadata.json"),
    "w"
) as f:
    json.dump(metadata, f, indent=4)

print(json.dumps(metadata, indent=4))

In [ ]:
print("Files saved in Drive:")

for file in os.listdir(SAVE_PATH):
    print(file)

### Conclusion

In this notebook, the Plant Disease dataset was downloaded using **KaggleHub** and prepared for the proposed Explainable AI-based plant disease detection system. To reduce computational complexity, the project was focused on two classes: **Tomato Early Blight** and **Tomato Healthy**.

The dataset contained **1,000 Tomato Early Blight images** and **1,591 Tomato Healthy images**. To maintain a balanced dataset, 1,000 healthy images were randomly selected, resulting in a total of **2,000 images** with equal representation of both classes.

The dataset was divided using stratified sampling into **70% training, 15% validation, and 15% testing**, resulting in 1,400 training images, 300 validation images, and 300 testing images.

For preprocessing, all images were converted to RGB format, resized to **224 × 224 pixels**, converted into PyTorch tensors, and normalized using the standard ImageNet mean and standard deviation values. Each processed image has the tensor shape **3 × 224 × 224**.

The processed datasets were saved as PyTorch `.pt` files on Google Drive along with a metadata file containing the class mapping, image dimensions, normalization parameters, and dataset split information.

The final prepared dataset will be used in the next notebook for **transfer-learning-based disease classification using the T4 GPU**. The trained model will subsequently serve as the foundation for the project's **Grad-CAM explainability, trust-score calculation, RAG, LLM explanation, and human-in-the-loop feedback components**.